# 03 Author and edit an archetype

This notebook builds a small archetype from explicit materials, edits archetypes
without changing the originals, saves one as plain data and exports the constructions
that EPSM Core reads.

**Prerequisites:** `epsm_data` installed. Runs offline. The two materials below are
**illustrative values, not sourced data**. Thicknesses are in metres, conductivity in
W/(m K), density in kg/m³ and specific heat in J/(kg K).

Define two materials. Each needs its thermal and surface properties.

In [ ]:
from epsm_data import Material, Roughness

brick = Material(
    name="Example brick", json_id="example:brick", boverket_id=None,
    material_type="EnergyMaterial", roughness=Roughness.MEDIUM_ROUGH,
    density=1650, specific_heat=800, conductivity=0.6,
    thermal_absorptance=0.9, solar_absorptance=0.7, visible_absorptance=0.7,
)
wool = brick.with_changes(
    name="Example mineral wool", json_id="example:wool",
    density=30, specific_heat=840, conductivity=0.037,
)

Describe the archetype as a recipe and build it. Layers run from outside to inside;
the window is a whole-window U-value and g-value.

In [ ]:
from epsm_data import build_archetype

spec = {
    "archetype_id": "example_mf_1970",
    "display_name": "Example multi-family, 1970",
    "construction_year": 1970,
    "building_typology": "multi_family",
    "elements": {
        "EXT_WALL": {"components": [
            {"shearing_layer": "Skin", "layers": [{"material_id": "brick", "thickness": 0.12}]},
            {"shearing_layer": "Structure", "layers": [{"material_id": "wool", "thickness": 0.10}]},
        ]},
        "WINDOW": {"u_value": 2.5, "shgc": 0.7},
    },
}
example = build_archetype(spec, materials={"brick": brick, "wool": wool})
example

BuildingArchetype(id='example_mf_1970', elements=2, missing=3)

The printout says three sections are missing: no operating profile, ventilation or
envelope performance was supplied. That is allowed; Core checks what it needs.

In [ ]:
print(example.missing_sections)
print({role: round(u, 3) for role, u in example.envelope_u_values.items()})

('operating_profile', 'hvac_template', 'envelope_performance')
{'WINDOW': 2.5, 'wall': 0.325}


Check the wall by hand: surface resistances 0.13 + 0.04 plus each layer's d/λ.

In [ ]:
expected = 1 / (0.13 + 0.12 / 0.6 + 0.10 / 0.037 + 0.04)
assert abs(example.envelope_u_values["wall"] - expected) < 1e-9
print(round(expected, 3))

0.325


Every definition is immutable. `with_changes` returns a new copy; `with_element`
swaps one role in a construction set. Here the window is replaced.

In [ ]:
new_window = example.window.with_changes(u_value=1.0, shgc=0.5)
renovated = example.with_changes(
    display_name="Example multi-family, new windows",
    construction_set=example.construction_set.with_element("WINDOW", new_window),
)
print("original:", example.window.u_value, "| edited:", renovated.window.u_value)

original: 2.5 | edited: 1.0


The same works on a catalogue archetype, for example a tighter envelope after air sealing.

In [ ]:
from epsm_data import Catalogue

catalogue = Catalogue.load()
selected = catalogue.get("mf_se02", state="existing")
sealed = selected.with_changes(
    display_name="MF SE.02, air sealed",
    envelope_performance=selected.envelope_performance.with_changes(infiltration_l_s_m2=0.1),
)
print(selected.envelope_performance)
print(sealed.envelope_performance)

EnvelopePerformance(l_s_m2=0.2, basis='Atemp', bridges=unknown, sources=0)
EnvelopePerformance(l_s_m2=0.1, basis='Atemp', bridges=unknown, sources=0)


`archetype_to_dict` writes the whole hierarchy, materials included, as JSON-ready data.
`archetype_from_dict` restores an equal archetype.

In [ ]:
import json

from epsm_data import archetype_from_dict, archetype_to_dict

text = json.dumps(archetype_to_dict(sealed))
restored = archetype_from_dict(json.loads(text))
assert restored == sealed
print(f"{len(text):,} characters of JSON; restored {restored.display_name!r}")

31,475 characters of JSON; restored 'MF SE.02, air sealed'


`export_nominal_constructions` produces the handover that EPSM Core reads: native
layer lists for Core and Honeybee, plus a nominal U-value per role.

In [ ]:
from epsm_data import export_nominal_constructions

bundle = export_nominal_constructions(example)
print(bundle["kind"], bundle["declared_roles"])
for layer in bundle["core"]["EXT_WALL"]["layers"]:
    print(f"  {layer['thickness_m']} m, λ = {layer['material']['conductivity_w_mk']}")

nominal_construction_handover ['EXT_WALL', 'WINDOW']
  0.12 m, λ = 0.6
  0.1 m, λ = 0.037


`validate_nominal_handover` checks the Core and Honeybee constructions describe the same physics.

In [ ]:
from epsm_data import validate_nominal_handover

validate_nominal_handover(bundle)

{'honeybee': True, 'core': True}

For a catalogue archetype, pass `nominal_metadata` so the bundle carries its type, era,
state, roof topology and material source references.

In [ ]:
metadata = catalogue.nominal_metadata("mf_se02", state="existing")
bundle = export_nominal_constructions(sealed, metadata=metadata)
print(bundle["key"], bundle["roof_topology"], validate_nominal_handover(bundle))
for role, values in bundle["roles"].items():
    print(f"  {role:<25} U = {values['nominal_u_w_m2k']:.2f}")

{'category': 'multi_family', 'era': 'SE.02', 'state': 'existing'} cold_attic {'honeybee': True, 'core': True}
  BASEMENT_WALL             U = 3.17
  EXT_WALL                  U = 0.42
  GROUND_FLOOR              U = 0.63
  INTERMEDIATE_FLOOR        U = 1.77
  INT_WALL_NONLOADBEARING   U = 0.60
  TOP_FLOOR                 U = 0.20
  WINDOW                    U = 2.22
